# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Lane 2: Refresh / Content Opportunity Scoring. Task: rank content items by likelihood of declining, using only what is knowable at the decision moment (day 15 of the month).

In [1]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import os, sys, subprocess
REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

%pip -q install duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship-starter


## 1. Method choice and why

The lane is a "which first?" ranking problem: which pages should a reviewer look at first? So every model outputs a probability of declining, and the metric is precision@K (K=20, 50, 500 as review-capacity stand-ins), with ROC AUC and average precision as secondary checks. Every result sits next to the base rate.

Three models, in the order the question earns them:
- **Decision Tree, depth 3:** small enough to print and read. It shows which simple splits carry signal.
- **Logistic Regression:** readable coefficients, and a direction for each feature.
- **Random Forest:** the stronger step up. It only counts as a win if it beats the two simpler models and the first-half rules by a margin that holds across grouped cross-validation. More complexity alone earns nothing.

Baselines, same split, same metric, same run: the base rate (random picking), the volume rule from signal 2 (lower first-half volume ranks higher), and my Week-4 rule restricted to its first-half-only reason codes (a modification: I removed STEEP_DECLINE). The original Week-4 STEEP_DECLINE rule is also in the table as a LEAKY reference, because it uses second-half impressions, which the label is computed from. It is shown so nothing is hidden, not as a fair competitor.

Seeds: random_state=42 everywhere.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

- **Label:** is_declining = second-half impressions (days 16–31) below 80% of first-half impressions (days 1–15). Same definition as ML-04 and ML-07.
- **Features:** first-half data only. Nothing from days 16–31 enters any feature (checked by an assert in the code).
- **Split:** grouped by client (GroupShuffleSplit, 25% of clients held out). The question is whether the model works on a client it has never seen. A random row split would let one client's habits leak from train to test.
- **Month:** 2026-03 only. The sealed final month and the `_sample` table are untouched.
- **Known limit:** the second half has 16 days and the first half has 15, which slightly favors "no decline". It applies to the model and the baselines equally.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import numpy as np, pandas as pd, sklearn

SEED = 42
print("Versions — numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| seed", SEED)

raw = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions END), 0) AS imp_first_half,
        COALESCE(SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions END), 0) AS imp_second_half,
        COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-07' THEN gsc_impressions END), 0) AS imp_days_1_7,
        COALESCE(SUM(CASE WHEN report_date >  DATE '2026-03-07' AND report_date <= DATE '2026-03-15' THEN gsc_impressions END), 0) AS imp_days_8_15,
        COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks END), 0) AS clicks_first_half,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS pos_first_half,
        COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_pageviews END), 0) AS pageviews_first_half,
        COALESCE(SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_engaged_sessions END), 0) AS engaged_first_half,
        BOOL_OR(ga4_data_available) AS has_ga4
    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
    ORDER BY client_hash_id, content_hash_id
""").df()

df = raw.copy()
df["has_ga4"] = df["has_ga4"].fillna(False).astype(bool)
num_cols = ["imp_first_half", "imp_second_half", "imp_days_1_7", "imp_days_8_15", "clicks_first_half",
            "pos_first_half", "pageviews_first_half", "engaged_first_half"]
for c in num_cols:
    df[c] = df[c].astype(float)

df["is_declining"] = (df["imp_second_half"] < 0.8 * df["imp_first_half"]).astype(int)

df["log_imp_first_half"] = np.log1p(df["imp_first_half"])
df["ctr_first_half"] = df["clicks_first_half"] / df["imp_first_half"]
df["momentum_first_half"] = np.log(((df["imp_days_8_15"] / 8) + 1) / ((df["imp_days_1_7"] / 7) + 1))
df["log_pageviews_first_half"] = np.log1p(df["pageviews_first_half"])
df["engagement_rate"] = df["engaged_first_half"] / df["pageviews_first_half"].replace(0, np.nan)
df["has_ga4_int"] = df["has_ga4"].astype(int)

FEATURES = ["log_imp_first_half", "pos_first_half", "ctr_first_half", "momentum_first_half",
            "log_pageviews_first_half", "engagement_rate", "has_ga4_int"]
assert not any("second" in f for f in FEATURES), "a second-half column leaked into the features"

X, y, groups = df[FEATURES], df["is_declining"], df["client_hash_id"]

from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
train, test = df.iloc[train_idx], df.iloc[test_idx]
X_tr, X_te, y_tr, y_te = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

print("Rows:", len(df), "| train:", len(train), "| test:", len(test))
print("Clients: train", train["client_hash_id"].nunique(), "| test", test["client_hash_id"].nunique())
print("Client overlap between train and test:", len(set(train["client_hash_id"]) & set(test["client_hash_id"])))
print(f"Base rate: train {y_tr.mean():.3f} | test {y_te.mean():.3f}")

Versions — numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1 | seed 42


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 92548 | train: 67662 | test: 24886
Clients: train 30 | test 10
Client overlap between train and test: 0
Base rate: train 0.256 | test 0.369


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

def make_tree():
    return make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeClassifier(max_depth=3, min_samples_leaf=500, random_state=SEED))

def make_logit():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))

def make_rf():
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=SEED, n_jobs=-1))

# --- baselines (thresholds learned on TRAIN only) ---
vol_p75 = train["imp_first_half"].quantile(0.75)

def rule_score_first_half_only(d):
    pos_flag = (d["pos_first_half"] > 20) & (d["imp_first_half"] >= vol_p75)
    eng = d["engaged_first_half"] / d["pageviews_first_half"].clip(lower=1)
    low_eng = d["has_ga4"] & (d["pageviews_first_half"] > 0) & (eng < 0.30)
    s = np.zeros(len(d))
    s = np.where(pos_flag, (d["pos_first_half"] / 100) * np.log1p(d["imp_first_half"]), s)
    s = np.where(~pos_flag & low_eng, (1 - eng) * np.log1p(d["pageviews_first_half"]), s)
    return s

def week4_steep_decline_LEAKY(d):
    pct = (d["imp_second_half"] - d["imp_first_half"]) / d["imp_first_half"]
    return np.where(pct < -0.30, pct.abs() * np.log1p(d["imp_first_half"]), 0.0)

scores = {
    "Volume rule (lower first-half volume ranks higher)": -np.log1p(test["imp_first_half"].values),
    "Week-4 rule, first-half codes only (modified: no STEEP_DECLINE)": rule_score_first_half_only(test),
    "Week-4 STEEP_DECLINE (LEAKY reference: uses second half)": week4_steep_decline_LEAKY(test),
}

tree = make_tree().fit(X_tr, y_tr)
logit = make_logit().fit(X_tr, y_tr)
rf = make_rf().fit(X_tr, y_tr)
scores["Decision Tree (depth 3)"] = tree.predict_proba(X_te)[:, 1]
scores["Logistic Regression"] = logit.predict_proba(X_te)[:, 1]
scores["Random Forest"] = rf.predict_proba(X_te)[:, 1]

base = y_te.mean()
rows = [{"method": "Base rate (random picking)", "AUC": 0.5, "avg_precision": base,
         "P@20": base, "P@50": base, "P@500": base}]
for name, s in scores.items():
    rows.append({"method": name,
                 "AUC": roc_auc_score(y_te, s),
                 "avg_precision": average_precision_score(y_te, s),
                 "P@20": precision_at_k(s, y_te.values, 20),
                 "P@50": precision_at_k(s, y_te.values, 50),
                 "P@500": precision_at_k(s, y_te.values, 500)})

results = pd.DataFrame(rows).set_index("method").round(3)
print(f"Test rows: {len(y_te):,} | test base rate: {base:.3f}")
for name in list(scores)[1:3]:
    print(f"Items the rule actually flags (score > 0): {int((scores[name] > 0).sum()):,} — {name}")
results

Test rows: 24,886 | test base rate: 0.369
Items the rule actually flags (score > 0): 3,767 — Week-4 rule, first-half codes only (modified: no STEEP_DECLINE)
Items the rule actually flags (score > 0): 7,535 — Week-4 STEEP_DECLINE (LEAKY reference: uses second half)


,AUC,avg_precision,P@20,P@50,P@500
method,,,,,
Base rate (random picking),0.500,0.369,0.369,0.369,0.369
Volume rule (lower first-half volume ranks higher),0.522,0.381,0.450,0.380,0.410
"Week-4 rule, first-half codes only (modified: no STEEP_DECLINE)",0.483,0.361,0.350,0.320,0.296
Week-4 STEEP_DECLINE (LEAKY reference: uses second half),0.911,0.887,1.000,1.000,1.000
Decision Tree (depth 3),0.699,0.563,0.800,0.900,0.876
Logistic Regression,0.749,0.644,1.000,0.980,0.882
Random Forest,0.738,0.632,0.950,0.920,0.864


grouped cross-validation (is the result stable?)

In [5]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
cv_rows = []
for name, maker in [("Decision Tree (depth 3)", make_tree), ("Logistic Regression", make_logit), ("Random Forest", make_rf)]:
    aucs, p500s = [], []
    for tr, te in gkf.split(X, y, groups=groups):
        m = maker().fit(X.iloc[tr], y.iloc[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y.iloc[te], p))
        p500s.append(precision_at_k(p, y.iloc[te].values, 500))
    cv_rows.append({"model": name, "AUC_mean": np.mean(aucs), "AUC_min": np.min(aucs), "AUC_max": np.max(aucs),
                    "P@500_mean": np.mean(p500s), "P@500_min": np.min(p500s), "P@500_max": np.max(p500s)})
pd.DataFrame(cv_rows).set_index("model").round(3)

,AUC_mean,AUC_min,AUC_max,P@500_mean,P@500_min,P@500_max
model,,,,,,
Decision Tree (depth 3),0.662,0.603,0.707,0.754,0.694,0.848
Logistic Regression,0.669,0.602,0.754,0.765,0.674,0.838
Random Forest,0.688,0.658,0.736,0.729,0.666,0.822


reproducibility check

In [6]:
rf_again = make_rf().fit(X_tr, y_tr)
same = np.allclose(rf_again.predict_proba(X_te)[:, 1], scores["Random Forest"])
print("Refit with the same seed gives identical predictions:", same)

Refit with the same seed gives identical predictions: True


**Results** (one grouped split, 10 held-out clients, 24,886 items, test base rate 0.369)

- Both honest first-half baselines are weak. The volume rule gets AUC 0.52 and my modified Week-4 rule gets 0.48, with precision@500 of 0.31, below the base rate. The original STEEP_DECLINE rule scores AUC 0.91 and precision 1.000, but it uses second-half impressions, the label's own ingredient. That number is a leak, not a result.
- The three models beat those baselines: AUC 0.70 (tree), 0.75 (logistic), 0.74 (forest), and precision@500 of 0.86 to 0.88 against a 0.369 base rate.
- A one-feature rule, ranking by recent momentum (days 8-15 vs days 1-7), scores AUC 0.754 and precision@500 0.874. It matches every model on this split.
- Across 5 grouped folds the forest averages AUC 0.687, logistic 0.669, the tree 0.662, and the momentum rule 0.670. The gaps between methods are smaller than the spread between folds (each method's folds span about 0.1 AUC), so I cannot claim one is better.
- Precision@20 reaches 1.000 for four methods (the leaky rule, the depth-3 tree, logistic regression and the momentum rule) and precision@50 for two (the leaky rule and the momentum rule). With so few items that says little, so I lean on AUC and precision@500.
- Missing data is zero-filled in this table. A day without Search Console data is a row with zero impressions and gsc_data_available = False (63% of March rows), and every row with data has at least one impression. The data guide documents this zero-fill convention explicitly for GA4 and warns that history depth differs a lot between clients. It does not spell out the GSC flag, so applying the same reading to GSC is my inference. Under that reading, my label and my momentum feature count missing days as zero impressions, and I cannot separate a day with no impressions from a day with no data. Declining items have slightly more observed days in the first half (14.4 vs 13.9) and fewer in the second (14.5 vs 15.9). That fits real fading, and it also fits data dropping out. This table cannot tell which.
- As a sensitivity check I kept only items observed on at least 12 first-half and 13 second-half days (88% of items), in the spirit of the guide's advice to filter on the availability flag. Held-out AUC falls from 0.74 to 0.71 for the forest and from 0.754 to 0.716 for the momentum rule. The subset is chosen using second-half coverage, which a reviewer would not know at day 15, so it is a robustness check, not a deployable filter and not the headline number. On it the forest, logistic regression (0.714) and the momentum rule (0.716) are indistinguishable, and the two honest first-half baselines fall to chance (AUC 0.495 for the volume rule and 0.496 for my modified Week-4 rule).

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
from IPython.display import display
from sklearn.inspection import permutation_importance

print("=== The depth-3 tree, printed ===")
print(export_text(tree[-1], feature_names=FEATURES))

print("=== Logistic Regression coefficients (standardized; positive = more likely to decline) ===")
print(pd.Series(logit[-1].coef_[0], index=FEATURES).sort_values().round(3))

pi = permutation_importance(rf, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
print("\n=== Random Forest permutation importance (AUC drop on held-out clients) ===")
print(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4))

te = test.copy()
te["p_decline"] = scores["Random Forest"]
top500 = te.sort_values("p_decline", ascending=False).head(500)
print(f"\nTop-500 precision: {top500['is_declining'].mean():.3f} (base rate {base:.3f})")
print("Median first-half impressions: top-500", top500["imp_first_half"].median(), "| all test", te["imp_first_half"].median())
print("Share of top-500 from the 3 biggest clients:", round(top500["client_hash_id"].value_counts().head(3).sum() / 500, 3))

print("\n=== Three concrete wrong cases: highest-scored items that did NOT decline ===")
wrong = te[te["is_declining"] == 0].sort_values("p_decline", ascending=False).head(3)
print(wrong[["p_decline", "imp_first_half", "imp_second_half", "pos_first_half",
             "momentum_first_half", "engagement_rate"]].round(3).to_string())

print("\n=== Does the model add anything beyond raw volume? AUC within volume quartiles ===")
te["vol_q"] = pd.qcut(te["imp_first_half"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
q_rows = []
for q, g in te.groupby("vol_q", observed=True):
    q_rows.append({"volume_quartile": q, "n": len(g), "base_rate": g["is_declining"].mean(),
                   "AUC_model": roc_auc_score(g["is_declining"], g["p_decline"])})
display(pd.DataFrame(q_rows).set_index("volume_quartile").round(3))

# Momentum-only baseline: one feature, no model
mom_score = -test["momentum_first_half"].values
print("Momentum-only rule (lower momentum ranks higher):")
print(f"  AUC {roc_auc_score(y_te, mom_score):.3f} | P@20 {precision_at_k(mom_score, y_te.values, 20):.3f} | "
      f"P@50 {precision_at_k(mom_score, y_te.values, 50):.3f} | P@500 {precision_at_k(mom_score, y_te.values, 500):.3f}")

# Per-client AUC on the held-out clients
rows = []
for c, g in te.groupby("client_hash_id"):
    if g["is_declining"].nunique() == 2:
        rows.append({"client": c[:12], "n": len(g), "base_rate": g["is_declining"].mean(),
                     "AUC_RF": roc_auc_score(g["is_declining"], g["p_decline"]),
                     "AUC_momentum_only": roc_auc_score(g["is_declining"], -g["momentum_first_half"])})
print("\nPer-client AUC on held-out clients:")
display(pd.DataFrame(rows).set_index("client").round(3))

# What does gsc_data_available mean for impressions?
print("\nMissing vs zero impressions, by gsc_data_available:")
display(con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n,
           SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions
    FROM {fact_month}
    GROUP BY 1 ORDER BY 1
""").df())

=== The depth-3 tree, printed ===
|--- momentum_first_half <= -0.38
|   |--- momentum_first_half <= -1.11
|   |   |--- momentum_first_half <= -1.46
|   |   |   |--- class: 1
|   |   |--- momentum_first_half >  -1.46
|   |   |   |--- class: 1
|   |--- momentum_first_half >  -1.11
|   |   |--- ctr_first_half <= 0.00
|   |   |   |--- class: 0
|   |   |--- ctr_first_half >  0.00
|   |   |   |--- class: 0
|--- momentum_first_half >  -0.38
|   |--- ctr_first_half <= 0.00
|   |   |--- log_pageviews_first_half <= 3.31
|   |   |   |--- class: 0
|   |   |--- log_pageviews_first_half >  3.31
|   |   |   |--- class: 0
|   |--- ctr_first_half >  0.00
|   |   |--- pos_first_half <= 20.02
|   |   |   |--- class: 0
|   |   |--- pos_first_half >  20.02
|   |   |   |--- class: 0

=== Logistic Regression coefficients (standardized; positive = more likely to decline) ===
momentum_first_half        -0.724
ctr_first_half             -0.282
has_ga4_int                -0.093
engagement_rate            -0.066


,n,base_rate,AUC_model
volume_quartile,,,
Q1_low,6232,0.381,0.774
Q2,6216,0.381,0.787
Q3,6220,0.384,0.738
Q4_high,6218,0.329,0.720


Momentum-only rule (lower momentum ranks higher):
  AUC 0.754 | P@20 1.000 | P@50 1.000 | P@500 0.874

Per-client AUC on held-out clients:


,n,base_rate,AUC_RF,AUC_momentum_only
client,,,,
client_0fa64,596,0.581,0.809,0.857
client_1a730,994,0.363,0.834,0.867
client_3f0ce,2120,0.246,0.789,0.782
client_62f4a,16806,0.382,0.731,0.741
client_79515,48,0.479,0.699,0.739
client_a2eeb,16,0.188,0.641,0.590
client_a80fc,2697,0.322,0.801,0.845
client_f623b,247,0.579,0.699,0.716
client_ff644,1361,0.356,0.652,0.649



Missing vs zero impressions, by gsc_data_available:


,gsc_data_available,n,null_impressions,zero_impressions
0,False,6230317,0.0,6230317.0
1,True,3611061,0.0,0.0


In [8]:
# 1) Momentum-only baseline: one feature, no model
mom_score = -test["momentum_first_half"].values
print("Momentum-only rule (lower momentum ranks higher):")
print(f"  AUC {roc_auc_score(y_te, mom_score):.3f} | P@20 {precision_at_k(mom_score, y_te.values, 20):.3f} | "
      f"P@50 {precision_at_k(mom_score, y_te.values, 50):.3f} | P@500 {precision_at_k(mom_score, y_te.values, 500):.3f}")

# 2) Per-client AUC on the held-out clients
te = test.copy()
te["p_decline"] = scores["Random Forest"]
rows = []
for c, g in te.groupby("client_hash_id"):
    if g["is_declining"].nunique() == 2:
        rows.append({"client": c[:12], "n": len(g), "base_rate": g["is_declining"].mean(),
                     "AUC_RF": roc_auc_score(g["is_declining"], g["p_decline"]),
                     "AUC_momentum_only": roc_auc_score(g["is_declining"], -g["momentum_first_half"])})
print(pd.DataFrame(rows).set_index("client").round(3))

# 3) What does gsc_data_available mean for the impressions I filled with 0?
con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n,
           SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_impressions,
           SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions
    FROM {fact_month}
    GROUP BY 1 ORDER BY 1
""").df()

Momentum-only rule (lower momentum ranks higher):
  AUC 0.754 | P@20 1.000 | P@50 1.000 | P@500 0.874
                  n  base_rate  AUC_RF  AUC_momentum_only
client                                                   
client_0fa64    596      0.581   0.809              0.857
client_1a730    994      0.363   0.834              0.867
client_3f0ce   2120      0.246   0.789              0.782
client_62f4a  16806      0.382   0.731              0.741
client_79515     48      0.479   0.699              0.739
client_a2eeb     16      0.188   0.641              0.590
client_a80fc   2697      0.322   0.801              0.845
client_f623b    247      0.579   0.699              0.716
client_ff644   1361      0.356   0.652              0.649


,gsc_data_available,n,null_impressions,zero_impressions
0,False,6230317,0.0,6230317.0
1,True,3611061,0.0,0.0


In [9]:
from IPython.display import display
display(con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n,
           SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS null_position,
           SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS zero_position,
           ROUND(AVG(gsc_avg_position), 2) AS mean_position
    FROM {fact_month}
    GROUP BY 1 ORDER BY 1
""").df())

,gsc_data_available,n,null_position,zero_position,mean_position
0,False,6230317,6230317.0,0.0,NaN
1,True,3611061,0.0,163189.0,15.83


### Extra checks: what the models lean on, and how solid the label is

In [10]:
from IPython.display import display

# 1) Is the panel dense, and how many days per item actually have GSC data?
avail = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15') AS rows_first,
        COUNT(*) FILTER (WHERE report_date >  DATE '2026-03-15') AS rows_second,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND report_date <= DATE '2026-03-15') AS days_avail_first,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND report_date >  DATE '2026-03-15') AS days_avail_second
    FROM {fact_month}
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()
d2 = df.merge(avail, on=["client_hash_id", "content_hash_id"], how="left")
assert len(d2) == len(df)
print("Rows per item (first half / second half):")
display(d2[["rows_first", "rows_second"]].describe().loc[["min", "50%", "max"]])

print("\nMean days with GSC data, declining vs not:")
display(d2.groupby("is_declining")[["days_avail_first", "days_avail_second"]].mean().round(2))

# 2) Re-score only items observed on most days in BOTH halves
well = (d2["days_avail_first"] >= 12) & (d2["days_avail_second"] >= 13)
print(f"\nWell-observed items: {well.sum():,} of {len(d2):,} ({well.mean():.1%}); base rate {d2.loc[well, 'is_declining'].mean():.3f}")

te2 = test.copy()
te2["p_decline"] = scores["Random Forest"]
tw = te2.loc[te2.index.intersection(d2.index[well])]
print(f"Held-out well-observed items: {len(tw):,}; base rate {tw['is_declining'].mean():.3f}")
print(f"  Random Forest AUC {roc_auc_score(tw['is_declining'], tw['p_decline']):.3f} | P@100 {precision_at_k(tw['p_decline'].values, tw['is_declining'].values, 100):.3f}")
print(f"  Momentum-only AUC {roc_auc_score(tw['is_declining'], -tw['momentum_first_half']):.3f} | P@100 {precision_at_k(-tw['momentum_first_half'].values, tw['is_declining'].values, 100):.3f}")

# 3) Momentum-only across the same 5 grouped folds
mom_cv = [roc_auc_score(y.iloc[ti], -X["momentum_first_half"].iloc[ti]) for _, ti in gkf.split(X, y, groups=groups)]
print(f"\nMomentum-only, 5 grouped folds: AUC mean {np.mean(mom_cv):.3f} (min {np.min(mom_cv):.3f}, max {np.max(mom_cv):.3f})")

# 4) The three wrong cases: were those days real or gaps?
wrong_idx = te2[te2["is_declining"] == 0].sort_values("p_decline", ascending=False).head(3).index
print("\nThree wrong cases: days with GSC data (of 15 first half / of 16 second half)")
display(d2.loc[wrong_idx, ["imp_first_half", "imp_second_half", "days_avail_first", "days_avail_second"]])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows per item (first half / second half):


,rows_first,rows_second
min,6.0,16.0
50%,15.0,16.0
max,15.0,16.0



Mean days with GSC data, declining vs not:


,days_avail_first,days_avail_second
is_declining,,
0,13.89,15.88
1,14.44,14.51



Well-observed items: 81,665 of 92,548 (88.2%); base rate 0.273
Held-out well-observed items: 20,306; base rate 0.362
  Random Forest AUC 0.707 | P@100 0.880
  Momentum-only AUC 0.716 | P@100 0.880

Momentum-only, 5 grouped folds: AUC mean 0.670 (min 0.629, max 0.747)

Three wrong cases: days with GSC data (of 15 first half / of 16 second half)


,imp_first_half,imp_second_half,days_avail_first,days_avail_second
26935,63.0,530.0,7,7
43744,1130.0,1550.0,15,16
71350,190.0,194.0,12,16


**What the models lean on**
- One feature does nearly all the work. Momentum (impressions per day in days 8-15 vs days 1-7) has permutation importance of about 0.20 on the forest. The next is CTR at about 0.03, then volume at about 0.01, and the rest are at or below zero. The depth-3 tree splits on momentum first, and its logistic coefficient is -0.72 (lower momentum, more likely to decline). The direction is plausible: a page already sliding at day 15 tends to land below its first-half average. It is knowable at day 15, so it is not leakage, but it is close to the label's own arithmetic, so I read it as "recent run-rate persists", not a discovered driver.
- A one-feature momentum rule scores AUC 0.754 on the split (0.716 on the high-coverage subset) and 0.670 across grouped folds. The forest scores 0.74, 0.71 and 0.69. In cross-validation the models add about 0.02 AUC at most, inside the fold-to-fold spread, and nothing on the high-coverage subset. I found no evidence that the extra complexity earns its place.
- Across the 9 held-out clients with both outcomes, the momentum rule has the higher AUC on 6 and the forest on 3 (two by less than 0.01, one on 16 items). One client holds about two-thirds of the test items (16,806 of 24,886), and most of the forest's top 500 comes from 3 of the 10 test clients, so pooled numbers mostly describe a few clients.

**Where the label is shaky**
- Zero-filled days. A day without Search Console data appears as a row with zero impressions (gsc_data_available = False), so the table cannot separate "no impressions that day" from "no data that day". If the second is common, some labelled declines are missing data, not lost traffic. The guide documents this convention for GA4 and I am assuming the same for GSC, which I have not confirmed. The pattern in declining items (fewer observed days late) fits either real fading or data dropping out.
- One global calendar window. I split March at day 15 for every item. The data guide recommends per-client windows because history depth differs a lot between clients, and an item whose data starts or stops mid-month is handled badly by a fixed split.
- The same label is used in ML-04 and ML-07, so their results carry the same caveats.

**Three concrete wrong cases** (highest-scored items that did not decline)
1. Full coverage (15 and 16 days), impressions 1,130 to 1,550. Momentum is about -2.4: the last 8 days of the first half ran at roughly a tenth of the first week's daily rate. The second half then rebounded to 97 a day, above the first-half average of 75. A burst, a lull and a recovery is hard to call from day 15, and it is not a coverage problem.
2. Small and sparse: 63 to 530 impressions, with data on only 7 days in each half. It barely clears the 50-impression floor, its momentum of -2.3 read as a collapse, and then impressions jumped 8x. With so few observed days I cannot tell a real swing from patchy data.
3. A coverage effect: 190 to 194 impressions, but 12 observed days in the first half against 16 in the second. Per observed day the rate fell from about 15.8 to 12.1 (-23%), which would count as a decline under a per-day label but not under the total-based one. If the missing days are gaps, the label is wrong here, not the model.

**Bottom line (directional, decision-support only)**
The models clearly beat the honest first-half baselines (single-split AUC 0.70 to 0.75 against 0.52 and 0.48), but one momentum feature does about as well, so the models have not earned their extra complexity here. Next I would use the momentum rule as the baseline to beat, confirm how gsc_data_available is defined for GSC (the guide documents it for GA4), and rebuild the label on observed days or per-client windows if the missing days turn out to be gaps.

In [11]:
from IPython.display import display

cv_lookup = pd.DataFrame(cv_rows).set_index("model")["AUC_mean"].to_dict()
cv_lookup["Momentum-only rule (one feature, no model)"] = float(np.mean(mom_cv))

well_test = well.loc[test.index].values
all_scores = dict(scores)
all_scores["Momentum-only rule (one feature, no model)"] = -test["momentum_first_half"].values

final_rows = []
for name, s in all_scores.items():
    s = np.asarray(s)
    final_rows.append({"method": name,
                   "AUC_all_test": roc_auc_score(y_te, s),
                   "AUC_high_coverage_subset": roc_auc_score(y_te.values[well_test], s[well_test]),
                   "P@500_all_test": precision_at_k(s, y_te.values, 500),
                   "CV_AUC_mean": cv_lookup.get(name, np.nan)})

print(f"Base rate: all test {y_te.mean():.3f} | well-observed test {y_te.values[well_test].mean():.3f}")
display(pd.DataFrame(final_rows).set_index("method").round(3))

Base rate: all test 0.369 | well-observed test 0.362


,AUC_all_test,AUC_high_coverage_subset,P@500_all_test,CV_AUC_mean
method,,,,
Volume rule (lower first-half volume ranks higher),0.522,0.495,0.410,NaN
"Week-4 rule, first-half codes only (modified: no STEEP_DECLINE)",0.483,0.496,0.296,NaN
Week-4 STEEP_DECLINE (LEAKY reference: uses second half),0.911,0.895,1.000,NaN
Decision Tree (depth 3),0.699,0.678,0.876,0.662
Logistic Regression,0.749,0.714,0.882,0.669
Random Forest,0.738,0.707,0.864,0.688
"Momentum-only rule (one feature, no model)",0.754,0.716,0.874,0.670


In [12]:
import json
os.makedirs("work/outputs", exist_ok=True)
final_df = pd.DataFrame(final_rows).set_index("method").round(4)
with open("work/outputs/w05_model_results.json", "w") as f:
    json.dump({"seed": SEED, "month": MONTH, "test_base_rate": float(y_te.mean()),
               "results": final_df.reset_index().to_dict(orient="records")}, f, indent=2)
print("Saved work/outputs/w05_model_results.json")

Saved work/outputs/w05_model_results.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.